# Competition Summary Report

A supporter-focused overview of a complete Rugby Tracker competition season.

Set `COMPETITION` in the next cell, and set `SEASON` when the competition name is used for more than one season. Then run all cells. Names and seasons are matched case-insensitively.

In [ ]:
COMPETITION = "W6N"

# Example: "2026"; leave as None when the competition name is unique
SEASON = 2026

## Setup and data loading

In [ ]:
from __future__ import annotations

import os
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_rows", 100)
plt.style.use("seaborn-v0_8-whitegrid")

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "rugby_tracker").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from within the RugbyTracker repository.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DB_PATH = Path(os.environ.get("RUGBY_TRACKER_DB", REPO_ROOT / "data" / "rugbytracker.db")).expanduser()
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Rugby Tracker database not found: {DB_PATH}")

sys.path.insert(0, str(REPO_ROOT / "src"))
from rugby_tracker.standings import calculate_competition  # noqa: E402

connection = sqlite3.connect(f"file:{DB_PATH.resolve()}?mode=ro", uri=True)
connection.row_factory = sqlite3.Row
print(f"Using database: {DB_PATH}")

In [ ]:
competition_sql = "SELECT id, name, season, gender, ruleset FROM competitions WHERE name = ? COLLATE NOCASE"
params = [COMPETITION.strip()]
if SEASON is not None:
    competition_sql += " AND season = ? COLLATE NOCASE"
    params.append(str(SEASON).strip())
competitions = [dict(row) for row in connection.execute(competition_sql, params)]
if not competitions:
    available = pd.read_sql_query("SELECT name, season FROM competitions ORDER BY name, season", connection)
    raise ValueError(f"Competition not found: {COMPETITION!r} (season={SEASON!r}). Available:\n{available.to_string(index=False)}")
if len(competitions) > 1:
    seasons = ", ".join(str(row["season"]) for row in competitions)
    raise ValueError(f"Competition {COMPETITION!r} exists in multiple seasons ({seasons}); set SEASON.")
competition = competitions[0]
if not competition["ruleset"]:
    raise ValueError(f"{competition['name']} {competition['season']} has no standings ruleset.")

matches_sql = """
SELECT m.*, h.name AS home_team_name, hc.name AS home_team_country,
       a.name AS away_team_name, ac.name AS away_team_country,
       v.name AS venue_name
FROM matches m
JOIN teams h ON h.id = m.home_team_id
JOIN countries hc ON hc.id = h.country_id
JOIN teams a ON a.id = m.away_team_id
JOIN countries ac ON ac.id = a.country_id
LEFT JOIN venues v ON v.id = m.venue_id
WHERE m.competition_id = ?
ORDER BY m.match_date, COALESCE(m.kickoff_time, ''), m.id
"""
all_matches = [dict(row) for row in connection.execute(matches_sql, (competition["id"],))]
if not all_matches:
    raise ValueError(f"No scheduled matches found for {competition['name']} {competition['season']}.")
completed_matches = [m for m in all_matches if m["home_score"] is not None and m["away_score"] is not None]
if not completed_matches:
    raise ValueError(f"No completed matches found for {competition['name']} {competition['season']}.")
team_names = sorted({m["home_team_name"] for m in all_matches} | {m["away_team_name"] for m in all_matches}, key=str.casefold)
competition_result = calculate_competition(all_matches, competition["ruleset"])

In [ ]:
def match_view(match: dict) -> dict:
    home_score = int(match["home_score"])
    away_score = int(match["away_score"])
    home_tries = int(match["home_tries"]) if match["home_tries"] is not None else 0
    away_tries = int(match["away_tries"]) if match["away_tries"] is not None else 0
    outcome = "Home win" if home_score > away_score else "Away win" if away_score > home_score else "Draw"
    return {
        "Date": pd.to_datetime(match["match_date"]).date(),
        "Round": match["round"] or "—",
        "Home team": match["home_team_name"],
        "Away team": match["away_team_name"],
        "Final score": f"{home_score}–{away_score}",
        "Home points": home_score, "Away points": away_score,
        "Home tries": home_tries, "Away tries": away_tries,
        "Total points": home_score + away_score,
        "Total tries": home_tries + away_tries,
        "Winning margin": abs(home_score - away_score),
        "Outcome": outcome,
    }

results = pd.DataFrame(match_view(match) for match in completed_matches)
played = len(results)
home_wins = int((results["Outcome"] == "Home win").sum())
away_wins = int((results["Outcome"] == "Away win").sum())
draws = int((results["Outcome"] == "Draw").sum())

def cards(items):
    boxes = "".join(f"<div style='padding:12px 16px;border:1px solid #ddd;border-radius:8px;min-width:125px'><small>{label}</small><br><b style='font-size:1.25em'>{value}</b></div>" for label, value in items)
    display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap;margin:8px 0 18px'>{boxes}</div>"))

def match_description(row):
    return f"{row['Home team']} {row['Final score']} {row['Away team']} ({row['Date']:%d %b %Y})"

## Competition overview

In [ ]:
display(HTML(f"<h2>{competition['name']}</h2><p><b>Season {competition['season']}</b></p>"))
cards([
    ("Teams", len(team_names)), ("Completed matches", played),
    ("Scheduled matches", len(all_matches)), ("Total tries", int(results["Total tries"].sum())),
    ("Total points", int(results["Total points"].sum())),
])

## Final league table

In [ ]:
league_table = pd.DataFrame(competition_result["table"])[["Pos", "Team", "P", "W", "D", "L", "PF", "PA", "PD", "Pts"]]
league_table.columns = ["Position", "Team", "Played", "Won", "Drawn", "Lost", "Points For", "Points Against", "Points Difference", "Competition Points"]
display(league_table.style.hide(axis="index"))
if competition_result["validation_errors"]:
    display(HTML("<p><i>Table is provisional: " + "; ".join(competition_result["validation_errors"]) + "</i></p>"))

## Competition statistics

In [ ]:
highest = results.loc[results["Total points"].idxmax()]
lowest = results.loc[results["Total points"].idxmin()]
largest_margin = results.loc[results["Winning margin"].idxmax()]
cards([
    ("Average points / match", f"{results['Total points'].mean():.1f}"),
    ("Average tries / match", f"{results['Total tries'].mean():.1f}"),
    ("Home wins", home_wins), ("Away wins", away_wins), ("Draws", draws),
])
highlights = pd.DataFrame([
    ("Highest-scoring match", match_description(highest), int(highest["Total points"])),
    ("Lowest-scoring match", match_description(lowest), int(lowest["Total points"])),
    ("Largest winning margin", match_description(largest_margin), int(largest_margin["Winning margin"])),
], columns=["Highlight", "Match", "Points / margin"]).set_index("Highlight")
display(highlights)

## Team rankings

In [ ]:
full_table = pd.DataFrame(competition_result["table"])
ranking_specs = [
    ("Most competition points", "Pts", False),
    ("Most points scored", "PF", False),
    ("Best defence", "PA", True),
    ("Most tries scored", "TF", False),
    ("Fewest tries conceded", "TA", True),
]
rankings = []
for category, column, ascending in ranking_specs:
    ordered = full_table.sort_values([column, "Team"], ascending=[ascending, True], kind="stable")
    best_value = ordered.iloc[0][column]
    leaders = ", ".join(ordered.loc[ordered[column] == best_value, "Team"])
    rankings.append((category, leaders, int(best_value)))
display(pd.DataFrame(rankings, columns=["Category", "Leader", "Value"]).set_index("Category"))

## Home and away performance

In [ ]:
outcome_percentages = pd.Series({
    "Home wins": home_wins / played * 100,
    "Away wins": away_wins / played * 100,
    "Draws": draws / played * 100,
})
cards([(label, f"{value:.1f}%") for label, value in outcome_percentages.items()])
fig, ax = plt.subplots(figsize=(7.5, 3.8))
outcome_percentages.plot.bar(ax=ax, color=["#4c78a8", "#f58518", "#9e9e9e"])
ax.set(title="Match outcomes", xlabel="Outcome", ylabel="Percentage of completed matches", ylim=(0, 100))
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

## Scoring distribution and winning margins

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].hist(results["Total points"], bins="auto", edgecolor="white", color="#4c78a8")
axes[0].set(title="Points scored per match", xlabel="Total match points", ylabel="Matches")
axes[1].hist(results["Winning margin"], bins="auto", edgecolor="white", color="#f58518")
axes[1].set(title="Winning margins", xlabel="Points (draws = 0)", ylabel="Matches")
plt.tight_layout(); plt.show()

## Results by round

In [ ]:
round_summary = (results.groupby("Round", sort=False)
                 .agg(**{"Matches played": ("Final score", "size"), "Total tries": ("Total tries", "sum"), "Total points": ("Total points", "sum"), "Average points per game": ("Total points", "mean")})
                 .reset_index())
round_summary["Average points per game"] = round_summary["Average points per game"].round(1)
display(round_summary.style.hide(axis="index"))

fig, ax = plt.subplots(figsize=(9, 3.8))
x = range(1, len(round_summary) + 1)
ax.plot(x, round_summary["Average points per game"], marker="o", linewidth=2)
ax.set(title="Average points per match by round", xlabel="Round", ylabel="Average points", xticks=list(x), xticklabels=round_summary["Round"].astype(str))
ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## Highest-scoring matches

In [ ]:
highest_scoring = results.sort_values(["Total points", "Date"], ascending=[False, True], kind="stable")
display(highest_scoring[["Date", "Round", "Home team", "Away team", "Final score", "Total points"]].head(10).style.hide(axis="index"))

## Closest matches

In [ ]:
closest = results.sort_values(["Winning margin", "Date"], ascending=[True, True], kind="stable")
display(closest[["Date", "Home team", "Away team", "Final score", "Winning margin"]].head(10).style.hide(axis="index"))

---
**Notes.** Match statistics use completed fixtures only. Scheduled matches include every fixture stored for the selected competition. The league table uses the competition ruleset stored by Rugby Tracker and may exclude knockout rounds. Try totals treat missing try values in completed matches as zero. Tied highlights use the first match chronologically. The notebook is designed for standard Jupyter HTML or PDF export.